<a href="https://colab.research.google.com/github/fralfaro/ICS40125/blob/main/docs/labs/lab_03.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# ICS40125 - Laboratorio N°03





**Objetivo**: Aplicar técnicas avanzadas de manipulación y análisis de datos con pandas sobre un conjunto real de datos de contenido de Netflix, reforzando buenas prácticas y métodos eficientes sin recurrir a `groupby`, `merge`, `pivot`, ni `join`.



**Dataset**:

Trabajaremos con el archivo `netflix_titles.csv`, que contiene información sobre los títulos disponibles en la plataforma Netflix hasta el año 2021.

| Variable       | Clase     | Descripción                                                                 |
|----------------|-----------|------------------------------------------------------------------------------|
| show_id        | caracter  | Identificador único del título en el catálogo de Netflix.                   |
| type           | caracter  | Tipo de contenido: 'Movie' o 'TV Show'.                                     |
| title          | caracter  | Título del contenido.                                                       |
| director       | caracter  | Nombre del director (puede ser nulo).                                       |
| cast           | caracter  | Lista de actores principales (puede ser nulo).                              |
| country        | caracter  | País o países donde se produjo el contenido.                                |
| date_added     | fecha     | Fecha en la que el título fue agregado al catálogo de Netflix.              |
| release_year   | entero    | Año de lanzamiento original del título.                                     |
| rating         | caracter  | Clasificación por edad (por ejemplo: 'PG-13', 'TV-MA').                      |
| duration       | caracter  | Duración del contenido (minutos o número de temporadas para series).        |
| listed_in      | caracter  | Categorías o géneros en los que está clasificado el contenido.              |
| description    | caracter  | Breve sinopsis del contenido.                                               |




In [2]:
import pandas as pd

# Cargar datos
df = pd.read_csv('https://raw.githubusercontent.com/fralfaro/ICS40125/main/docs/labs/data/netflix_titles.csv')
df.head()

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm..."
1,s2,TV Show,Blood & Water,NaN,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t..."
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",NaN,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...
3,s4,TV Show,Jailbirds New Orleans,NaN,NaN,NaN,"September 24, 2021",2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo..."
4,s5,TV Show,Kota Factory,NaN,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...



### Parte 1: Limpieza y preparación

1. Revisar y describir el dataset:

   * ¿Cuántas filas y columnas tiene?
   * ¿Qué tipos de datos hay?
   * ¿Cuántos valores nulos hay por columna?

2. Transformar la columna `date_added` a tipo fecha.

3. Crear columnas auxiliares con `assign`:

   * Año (`year_added`)
   * Mes (`month_added`)



In [3]:
#FIXME
# 1. Revisar y describir el dataset
print("--- Información general ---")
df.info()

print("\n--- Valores nulos por columna ---")
print(df.isnull().sum())

# 2. Transformar la columna date_added a tipo fecha
# .str.strip() elimina espacios en blanco fantasmas que puedan arruinar el parsing
df['date_added'] = pd.to_datetime(df['date_added'].str.strip(), format='%B %d, %Y', errors='coerce')

# 3. Crear columnas auxiliares con assign (year_added y month_added)
df = df.assign(
    year_added = df['date_added'].dt.year,
    month_added = df['date_added'].dt.month
)

# Visualizar el resultado preliminar
df[['title', 'date_added', 'year_added', 'month_added']].head()

--- Información general ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8807 entries, 0 to 8806
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   show_id       8807 non-null   object
 1   type          8807 non-null   object
 2   title         8807 non-null   object
 3   director      6173 non-null   object
 4   cast          7982 non-null   object
 5   country       7976 non-null   object
 6   date_added    8797 non-null   object
 7   release_year  8807 non-null   int64 
 8   rating        8803 non-null   object
 9   duration      8804 non-null   object
 10  listed_in     8807 non-null   object
 11  description   8807 non-null   object
dtypes: int64(1), object(11)
memory usage: 825.8+ KB

--- Valores nulos por columna ---
show_id            0
type               0
title              0
director        2634
cast             825
country          831
date_added        10
release_year       0
rating             4


,title,date_added,year_added,month_added
0,Dick Johnson Is Dead,2021-09-25,2021.0,9.0
1,Blood & Water,2021-09-24,2021.0,9.0
2,Ganglands,2021-09-24,2021.0,9.0
3,Jailbirds New Orleans,2021-09-24,2021.0,9.0
4,Kota Factory,2021-09-24,2021.0,9.0


## Parte 2: Técnicas avanzadas de pandas

4. Utilizar `.loc` para seleccionar películas (`type == 'Movie'`) que fueron agregadas después del año 2018.

5. Utilizar `str.contains()` y `str.extract()`:

   * Filtrar títulos que contienen la palabra 'love' (sin distinguir mayúsculas/minúsculas).
   * Extraer la duración en minutos para las películas desde la columna `duration`.

6. Aplicar `explode()` sobre la columna `listed_in` para obtener una fila por cada género.

7. Obtener un top 10 de géneros más frecuentes utilizando `value_counts()`.

8. Aplicar `where()` y `mask()` para marcar las películas de más de 120 minutos como contenido largo en una nueva columna.

9. Utilizar `.loc` para filtrar películas que cumplen con:

   * Más de 100 minutos de duración.
   * Rating igual a `'R'`.
   * País igual a `'United States'`.

10. Utilizar `.style` para formatear visualmente el top 10 de películas más largas.

In [7]:
#FIXME # 4. Utilizar .loc para seleccionar películas agregadas después de 2018
peliculas_post_2018 = df.loc[(df['type'] == 'Movie') & (df['year_added'] > 2018)]

# 5. Utilizar str.contains() y str.extract()
# Filtrar títulos con la palabra 'love' (case=False ignora mayúsculas/minúsculas)
contiene_love = df[df['title'].str.contains('love', case=False, na=False)]

# Extraer la duración numérica en minutos desde la columna duration
df['duration_num'] = df['duration'].str.extract(r'(\d+)').astype(float)

# 6. Aplicar explode() sobre la columna listed_in para obtener una fila por género
# Primero separamos los strings por coma para convertirlos en listas
df_exploded = df.copy()
df_exploded['genre'] = df_exploded['listed_in'].str.split(', ')
df_exploded = df_exploded.explode('genre')

# 7. Obtener un top 10 de géneros más frecuentes
top_10_generos = df_exploded['genre'].value_counts().head(10)
print("--- Top 10 Géneros ---")
print(top_10_generos)

# 8. Aplicar where() y mask() para marcar películas de más de 120 minutos
# Creamos una columna base con valor por defecto
df['tipo_duracion'] = 'Normal/Corto'

# mask cambia los valores donde la condición se cumple (True)
# where mantiene los valores donde es True y cambia donde es False.
# Usaremos mask para marcar como 'Largo' si es película y supera los 120 min.
condicion_larga = (df['type'] == 'Movie') & (df['duration_num'] > 120)
df['tipo_duracion'] = df['tipo_duracion'].mask(condicion_larga, 'Contenido Largo')

# 9. Filtrar películas: >100 min, Rating 'R' y País 'United States'
peliculas_especificas = df.loc[
    (df['type'] == 'Movie') &
    (df['duration_num'] > 100) &
    (df['rating'] == 'R') &
    (df['country'] == 'United States')
]

# 10. Utilizar .style para formatear visualmente el top 10 de películas más largas
top_10_largas = df[df['type'] == 'Movie'].sort_values(by='duration_num', ascending=False).head(10)
top_10_estilizado = top_10_largas[['title', 'duration_num', 'release_year']].style.background_gradient(
    subset=['duration_num'], cmap='Blues'
).format({'duration_num': '{:.0f} min'})

# Para desplegar en Jupyter Notebook simplemente invoca:
top_10_estilizado

--- Top 10 Géneros ---
genre
International Movies        2752
Dramas                      2427
Comedies                    1674
International TV Shows      1351
Documentaries                869
Action & Adventure           859
TV Dramas                    763
Independent Movies           756
Children & Family Movies     641
Romantic Movies              616
Name: count, dtype: int64


,title,duration_num,release_year
4253,Black Mirror: Bandersnatch,312 min,2018
717,Headspace: Unwind Your Mind,273 min,2021
2491,The School of Mischief,253 min,1973
2487,No Longer kids,237 min,1979
2484,Lock Your Girls In,233 min,1982
2488,Raya and Sakina,230 min,1984
166,Once Upon a Time in America,229 min,1984
7932,Sangam,228 min,1964
1019,Lagaan,224 min,2001
4573,Jodhaa Akbar,214 min,2008




### Pregunta Desafío

11. ¿Cuáles son las combinaciones más frecuentes de género y rating en el dataset?
    (Sugerencia: utilizar `value_counts` con `subset=["genre", "rating"]` después de aplicar `explode()`).



### Bonus: Análisis de duplicados y limpieza

12. ¿Existen películas con el mismo nombre (`title`) pero con distinto año de lanzamiento (`release_year`)?
13. ¿Cuántos títulos únicos hay en total en la columna `title`?





In [9]:
#FIXME  # 11. Combinaciones más frecuentes de género y rating (usando el dataframe explotado)
combinaciones_frecuentes = df_exploded['genre'].value_counts() # Base para verificar si se requiere subset
# Utilizando subset como lo sugiere explícitamente el enunciado
top_combinaciones = df_exploded.value_counts(subset=['genre', 'rating']).head(10)
print("--- Top Combinaciones Género y Rating ---")
print(top_combinaciones)

print("\n-----------------------------------------\n")

# 12. ¿Existen películas con el mismo nombre pero con distinto año de lanzamiento?
# Buscamos duplicados en la columna 'title' pero manteniendo registros únicos por año de lanzamiento
titulos_años = df[['title', 'release_year']].drop_duplicates()
duplicados_distinto_ano = titulos_años[titulos_años.duplicated(subset=['title'], keep=False)].sort_values(by='title')

if not duplicados_distinto_ano.empty:
    print(f"Sí, existen {duplicados_distinto_ano['title'].nunique()} títulos con diferentes años de lanzamiento. Ejemplos:")
    print(duplicados_distinto_ano.head(6))
else:
    print("No existen películas con el mismo nombre y distinto año de lanzamiento.")

# 13. ¿Cuántos títulos únicos hay en total en la columna title?
titulos_unicos = df['title'].nunique()
print(f"\nCantidad total de títulos únicos en el dataset: {titulos_unicos}")

--- Top Combinaciones Género y Rating ---
genre                   rating
International Movies    TV-MA     1130
                        TV-14     1065
Dramas                  TV-MA      830
International TV Shows  TV-MA      714
Dramas                  TV-14      693
International TV Shows  TV-14      472
Comedies                TV-14      465
TV Dramas               TV-MA      434
Comedies                TV-MA      431
Dramas                  R          375
Name: count, dtype: int64

-----------------------------------------

No existen películas con el mismo nombre y distinto año de lanzamiento.

Cantidad total de títulos únicos en el dataset: 8807
